In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("../../data/02_processed/processed_emails.csv")

df.columns = df.columns.str.lower().str.strip()
df.head()

/var/folders/6q/3_ys7c717hz_scqx9xw7q35w0000gn/T/ipykernel_7788/3610029970.py:1: DtypeWarning: Columns (16,19) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../../data/02_processed/processed_emails.csv")


,co_ref,time_to_renewal,crm_accreditation_completed,crm_timely_completion,crm_progress_towards_accreditation,crm_delays_in_accreditation,crm_contractor_suggested_leave,crm_contractor_engagement,crm_contractor_sentiment,crm_contractor_sentiment_score,...,crm_accreditation_issues,crm_membership_overdue,crm_auto_renewal_status,crm_dissatisified_with_renewal_price,crm_customer_complained,crm_refund_mentioned,crm_negative_customer_experience,crm_dissatisfaction_with_support,crm_financial_hardship_mentioned,year
0,KG5766,pre_renewal,Not Discussed,Not Discussed,Not Discussed,Yes,No,Yes,Neutral,50,...,Not Discussed,Yes,0,No,No,Yes,Yes,No,Yes,2025
1,EJ1532,14_out,Not Discussed,Not Discussed,Not Discussed,No,Not Discussed,No,Not Discussed,Not Discussed,...,Not Discussed,Not Discussed,0,Not Discussed,No,Yes,Yes,No,Not Discussed,2025
2,AA4063,prior_year,Not Discussed,Not Discussed,Not Discussed,No,No,Yes,Neutral,50,...,Not Discussed,No,0,No,No,Yes,Yes,Yes,Not Discussed,2025
3,JY9888,prior_year,No,No,Not Discussed,Yes,No,Yes,Satisfied,80,...,Not Discussed,Yes,0,Not Discussed,No,Yes,Yes,No,Not Discussed,2025
4,WO6689,pre_renewal,Not Discussed,Not Discussed,Not Discussed,No,No,Yes,Satisfied,80,...,No,No,0,No,No,Yes,Yes,No,Not Discussed,2026


In [4]:
print(df.columns.tolist())

['co_ref', 'time_to_renewal', 'crm_accreditation_completed', 'crm_timely_completion', 'crm_progress_towards_accreditation', 'crm_delays_in_accreditation', 'crm_contractor_suggested_leave', 'crm_contractor_engagement', 'crm_contractor_sentiment', 'crm_contractor_sentiment_score', 'crm_dts_or_ssip_mentioned', 'crm_customer_payment_intention', 'crm_competitors_mentioned', 'crm_membership_level', 'crm_platform_issues_raised', 'crm_agent_chased_contractor', 'crm_agent_chase_count', 'crm_accreditation_issues', 'crm_membership_overdue', 'crm_auto_renewal_status', 'crm_dissatisified_with_renewal_price', 'crm_customer_complained', 'crm_refund_mentioned', 'crm_negative_customer_experience', 'crm_dissatisfaction_with_support', 'crm_financial_hardship_mentioned', 'year']


In [ ]:
bill_df = pd.read_csv("../../data/02_processed/processed_billings.csv")
bill_df.columns = bill_df.columns.str.lower().str.strip()

bill_df['prospect_renewal_date'] = pd.to_datetime(
    bill_df['prospect_renewal_date'], errors='coerce'
)

renewal_map = bill_df[['co_ref', 'prospect_renewal_date']].drop_duplicates()

df = df.merge(renewal_map, on='co_ref', how='left')

/var/folders/6q/3_ys7c717hz_scqx9xw7q35w0000gn/T/ipykernel_7788/3797275103.py:1: DtypeWarning: Columns (14,22) have mixed types. Specify dtype option on import or set low_memory=False.
  bill_df = pd.read_csv("../../data/02_processed/processed_billings.csv")


In [6]:
df = df.dropna(subset=['prospect_renewal_date'])

In [10]:
df['is_14_days_before'] = (df['time_to_renewal'] == '14_out').astype(int)
df['is_pre_renewal'] = (df['time_to_renewal'] == 'pre_renewal').astype(int)

In [11]:
agg_df = df.groupby('co_ref').agg({
    'time_to_renewal': 'count'
}).reset_index()

agg_df.rename(columns={'time_to_renewal': 'total_interactions'}, inplace=True)

In [12]:
agg_df = agg_df.merge(
    df.groupby('co_ref')['is_14_days_before'].sum().rename('interactions_14_days'),
    on='co_ref', how='left'
)

agg_df = agg_df.merge(
    df.groupby('co_ref')['is_pre_renewal'].sum().rename('pre_renewal_interactions'),
    on='co_ref', how='left'
)

In [13]:
agg_df['last_moment_engagement_ratio'] = (
    agg_df['interactions_14_days'] / (agg_df['total_interactions'] + 1)
)

In [14]:
agg_df = agg_df.merge(
    df.groupby('co_ref')['crm_customer_complained'].sum().rename('complaints'),
    on='co_ref', how='left'
)

agg_df = agg_df.merge(
    df.groupby('co_ref')['crm_negative_customer_experience'].sum().rename('negative_experience'),
    on='co_ref', how='left'
)

agg_df = agg_df.merge(
    df.groupby('co_ref')['crm_dissatisfaction_with_support'].sum().rename('support_issues'),
    on='co_ref', how='left'
)

In [15]:
agg_df = agg_df.merge(
    df.groupby('co_ref')['crm_financial_hardship_mentioned'].sum().rename('financial_stress'),
    on='co_ref', how='left'
)

agg_df = agg_df.merge(
    df.groupby('co_ref')['crm_dissatisified_with_renewal_price'].sum().rename('price_dissatisfaction'),
    on='co_ref', how='left'
)

In [19]:
df['crm_contractor_sentiment_score'] = pd.to_numeric(
    df['crm_contractor_sentiment_score'], errors='coerce'
)
df = df.dropna(subset=['crm_contractor_sentiment_score'])

In [24]:
agg_df = agg_df.drop(columns=['avg_sentiment'], errors='ignore')

agg_df = agg_df.merge(
    df.groupby('co_ref')['crm_contractor_sentiment_score']
      .mean()
      .rename('avg_sentiment'),
    on='co_ref', how='left'
)

In [26]:
df['crm_agent_chase_count'] = pd.to_numeric(
    df['crm_agent_chase_count'], errors='coerce'
)

df['crm_contractor_engagement'] = pd.to_numeric(
    df['crm_contractor_engagement'], errors='coerce'
)

In [27]:
df['crm_agent_chase_count'] = df['crm_agent_chase_count'].fillna(0)
df['crm_contractor_engagement'] = df['crm_contractor_engagement'].fillna(0)

In [28]:
agg_df = agg_df.merge(
    df.groupby('co_ref')['crm_agent_chase_count'].sum().rename('agent_followups'),
    on='co_ref', how='left'
)

agg_df = agg_df.merge(
    df.groupby('co_ref')['crm_contractor_engagement'].sum().rename('engagement_score'),
    on='co_ref', how='left'
)

In [29]:
agg_df = agg_df.fillna(0)

In [31]:
agg_df.to_csv("../../data/03_final/final_email_features.csv", index=False)